In [1]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.9 -q

Found existing installation: mediapipe 0.10.9
Uninstalling mediapipe-0.10.9:
  Successfully uninstalled mediapipe-0.10.9


In [2]:
import mediapipe as mp
print("MediaPipe版本:", mp.__version__)

# 测试是否能导入solutions
try:
    from mediapipe import solutions
    print("solutions模块导入成功！")
    print("支持的模块:", dir(solutions))
except Exception as e:
    print("导入失败:", e)

MediaPipe版本: 0.10.9
solutions模块导入成功！
支持的模块: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'download_utils', 'drawing_styles', 'drawing_utils', 'face_detection', 'face_mesh', 'face_mesh_connections', 'hands', 'hands_connections', 'holistic', 'mediapipe', 'objectron', 'pose', 'pose_connections', 'selfie_segmentation']


In [1]:
# 1. 安装依赖（只需要运行一次）
!pip install opencv-python mediapipe==0.10.9 -q

# 2. 导入库
import cv2
import mediapipe as mp
import math

# 3. 初始化MediaPipe手部检测（修复后的写法）
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

# 4. 手部检测配置
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 5. 计算两点距离
def get_distance(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)

# 6. 手势判断函数
def judge_gesture(landmarks):
    thumb_tip = landmarks[4]
    index_tip = landmarks[8]
    middle_tip = landmarks[12]
    ring_tip = landmarks[16]
    pinky_tip = landmarks[20]
    wrist = landmarks[0]
    
    index_len = get_distance(index_tip, wrist)
    thumb_index_len = get_distance(thumb_tip, index_tip)

    if index_len > 0.2 and thumb_index_len > 0.15:
        return "食指伸出 - 移动光标"
    elif thumb_index_len < 0.08:
        return "拇食相触 - 左键单击"
    elif all(get_distance(tip, wrist) < 0.18 for tip in [index_tip, middle_tip, ring_tip, pinky_tip]):
        return "握拳 - 右键单击"
    else:
        return "未识别有效手势"

# 7. 摄像头调用（AI Studio适配）
cap = cv2.VideoCapture(0)
print("程序启动，按 Q 键退出")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)

    gesture_text = "无手部检测"
    if result.multi_hand_landmarks:
        for hand_lms in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(
                frame, hand_lms, mp_hands.HAND_CONNECTIONS,
                mp_draw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=3),
                mp_draw.DrawingSpec(color=(255,0,0), thickness=2)
            )
            gesture_text = judge_gesture(hand_lms.landmark)

    cv2.putText(frame, gesture_text, (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow("手势识别演示(AI Studio)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("程序已退出")

程序启动，按 Q 键退出
程序已退出


E0000 00:00:1781251248.648994   17109 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1781251248.649051   17109 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1781251248.649071   17109 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
[ WARN:0@0.971] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@0.972] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> 